# Macro Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option('display.max_columns', None)

In [ ]:
data_path = os.path.join("..", 'data')
macro_raw = pd.read_csv(os.path.join(data_path, 'macro_data.csv'), index_col=0)

In [ ]:
macro_raw.head()

In [ ]:
df = macro_raw.copy()

In [ ]:
def adf_kpss_verdict(series: pd.Series, regression="c", alpha=0.05) -> dict:
    """Return both p-values and a joint verdict."""
    s = series.dropna()
    adf_p  = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression=regression, nlags="auto")[1]

    if adf_p < alpha and kpss_p > alpha:
        verdict = "stationary"
    elif adf_p >= alpha and kpss_p <= alpha:
        verdict = "unit root"
    else:
        verdict = "inconclusive"
    return {"ADF p": adf_p, "KPSS p": kpss_p, "verdict": verdict}


level_results = pd.DataFrame(
    {code: adf_kpss_verdict(df[code]) for code in df.columns}
).T.round(4)
print("\nLevel-series stationarity tests")
print(level_results)

In [ ]:
TRANSFORM = {
    "industrial_production":   "dlog",
    "real_person_income":      "dlog",
    "unemployment_rate":       "diff",
    "initial_jobless_claims":  "dlog",
    "cpi":                     "dlog",
    "oil_price":               "dlog",
    "vix":                     "level",
    "credit_spread":           "level",
    "yield_curve_slope":       "level",
    "fed_funds_rate":          "diff",
    "consumer_sentiment":      "diff",
    "housing_starts":          "dlog",
    "m2_money_supply":         "dlog",
}

def transform(series: pd.Series, kind: str) -> pd.Series:
    if kind == "level":
        return series
    if kind == "diff":
        return series.diff()
    if kind == "dlog":
        return np.log(series).diff()
    raise ValueError(f"Unknown transform: {kind}")

# Apply
macro_stationary = pd.DataFrame(
    {col: transform(df[col], TRANSFORM[col]) for col in df.columns}
).dropna()

print(f"Transformed panel shape: {macro_stationary.shape}")
macro_stationary.head()

In [ ]:
trans_results = pd.DataFrame(
    {col: adf_kpss_verdict(macro_stationary[col]) for col in macro_stationary.columns}
).T.round(4)

print("\nPost-transformation stationarity tests")
print(trans_results)

In [ ]:
comparison = pd.concat(
    [
        level_results.rename(columns={
            "ADF p": "ADF_level", "KPSS p": "KPSS_level", "verdict": "verdict_level"
        }),
        trans_results.rename(columns={
            "ADF p": "ADF_trans", "KPSS p": "KPSS_trans", "verdict": "verdict_trans"
        }),
    ],
    axis=1
)
print("\nLevel vs transformed comparison")
comparison

In [ ]:
scaler = StandardScaler()
macro_scaled_array = scaler.fit_transform(macro_stationary)

macro_scaled = pd.DataFrame(
    macro_scaled_array,
    index=macro_stationary.index,
    columns=macro_stationary.columns,
)

print("Mean of each standardised variable (should be ~0):")
print(macro_scaled.mean().round(4))
print("\nStandard deviation of each standardised variable (should be ~1):")
print(macro_scaled.std().round(4))
print(f"\nFinal panel shape: {macro_scaled.shape}")
print(f"Date range: {macro_scaled.index.min()} to {macro_scaled.index.max()}")

In [ ]:
macro_scaled.to_csv(os.path.join(data_path, 'macro_clean.csv'))

# Asset Data

In [ ]:
price_raw = pd.read_csv(os.path.join(data_path, 'market_data.csv'), index_col="Date")

In [ ]:
df1 = price_raw.copy()

In [ ]:
df1 = df1.pct_change().dropna()

In [ ]:
df1.head()

In [ ]:
df1.to_csv(os.path.join(data_path, 'market_clean.csv'))